# 04 Resultados y predicciones

Objetivo: generar las predicciones finales y preparar archivos para decision.

Salidas:

- predicciones PT,
- predicciones PP,
- Excel final con dos hojas,
- plantilla de validacion experta,
- plantilla de stock actual, stock minimo y stock maximo.
- plantillas de variables exogenas calendario/producto,
- rangos de decision: minimo, maximo, confianza y revision.

Las predicciones usan el modelo ML mas estable entre validacion y test cuando `forecast_model` esta configurado como `best_ml_stable`.

In [ ]:
from pathlib import Path
import sys
from io import StringIO

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import Markdown, display

def find_project_root(start=None):
    current = Path(start or Path.cwd()).resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "config" / "config.yml").exists() and (candidate / "src").exists():
            return candidate
    raise FileNotFoundError("No se encontro la raiz del proyecto. Abre VS Code en la carpeta 'Avance 3'.")

ROOT = find_project_root()
sys.path.insert(0, str(ROOT / "src"))

pd.set_option("display.max_columns", 80)
pd.set_option("display.max_rows", 30)
plt.style.use("default")

from quickbooks_forecast.config import load_config

config = load_config()
reports_dir = config["resolved_paths"]["reports_dir"]
processed_dir = config["resolved_paths"]["processed_dir"]
raw_dir = config["resolved_paths"]["raw_dir"]
images_dir = ROOT / "notebooks" / "images"
images_dir.mkdir(parents=True, exist_ok=True)

def save_fig(name, extension="png", resolution=160):
    path_final = images_dir / f"{name}.{extension}"
    plt.tight_layout()
    plt.savefig(path_final, dpi=resolution, bbox_inches="tight")
    print(f"Figura guardada: {path_final}")

**Preparacion de predicciones finales:**

Este bloque prepara el entorno final para generar predicciones y reportes usando los modelos entrenados.

In [ ]:
from quickbooks_forecast.modeling import predict_all
from quickbooks_forecast.decision import build_decision_templates

predictions = predict_all(config)
decision_outputs = build_decision_templates(config)

print("PT:", predictions["PT"].shape)
print("PP:", predictions["PP"].shape)

**Generacion de salidas PT y PP:**

Aqui se generan las dos tablas finales solicitadas: predicciones para PT y predicciones para PP. Tambien se crean plantillas de apoyo para decision operativa.

In [ ]:
exog_report = reports_dir / "exogenous_variables_plan.md"
if exog_report.exists():
    display(Markdown(exog_report.read_text(encoding="utf-8")))

demo_report = reports_dir / "variables_exogenas_ecuador_demo.md"
if demo_report.exists():
    display(Markdown(demo_report.read_text(encoding="utf-8")))

replacement_results = reports_dir / "exogenous_replacement_results.md"
if replacement_results.exists():
    display(Markdown(replacement_results.read_text(encoding="utf-8")))

replacement_report = reports_dir / "replacement_readiness.md"
if replacement_report.exists():
    display(Markdown(replacement_report.read_text(encoding="utf-8")))

**Evidencia conceptual y exogenas:**

Este bloque documenta la parte conceptual: que variables exogenas se usaron, por que bajan el error y por que el escenario asistido no debe confundirse con validacion oficial sin datos reales.

In [ ]:
pred_pt = pd.read_csv(reports_dir / "predicciones_pt.csv")
pred_pp = pd.read_csv(reports_dir / "predicciones_pp.csv")

print("Modelo usado PT")
display(pred_pt["modelo_usado"].value_counts().reset_index())
print("Modelo usado PP")
display(pred_pp["modelo_usado"].value_counts().reset_index())

display(pred_pt.head(20))
display(pred_pp.head(20))

**Revision de tablas finales:**

Las predicciones finales incluyen el modelo usado y los campos de confianza. Estas tablas son la salida que podria revisar produccion cada mes.

In [ ]:
resumen_pred = pd.concat([pred_pt, pred_pp], ignore_index=True)
resumen = (
    resumen_pred.groupby(["source_type", "periodo", "modelo_usado"], as_index=False)
    .agg(productos=("product_id", "nunique"), cantidad_predicha=("cantidad_predicha", "sum"))
)
display(resumen)

fig, ax = plt.subplots(figsize=(12, 5))
for source, group in resumen_pred.groupby("source_type"):
    mensual = group.groupby("periodo")["cantidad_predicha"].sum()
    ax.plot(pd.to_datetime(mensual.index), mensual.values, marker="o", label=source)
ax.set_title("Prediccion mensual total")
ax.set_ylabel("Cantidad predicha")
ax.legend()
save_fig("04_prediccion_mensual_total")
plt.show()

**Lectura mensual agregada:**

El resumen mensual ayuda a ver la carga total esperada para PT y PP. Sirve como primera lectura gerencial antes de bajar al detalle por producto.

In [ ]:
print("Top PT por cantidad predicha")
display(pred_pt.sort_values("cantidad_predicha", ascending=False).head(30))

print("Top PP por cantidad predicha")
display(pred_pp.sort_values("cantidad_predicha", ascending=False).head(30))

**Productos criticos por volumen:**

Los productos top por cantidad predicha son los de mayor impacto operativo. Cualquier error en estos productos afecta mas al WAPE y a la produccion real.

In [ ]:
validation = pd.read_csv(ROOT / "data" / "input" / "validacion_expertos_template.csv")
stock = pd.read_csv(ROOT / "data" / "input" / "stock_min_max_template.csv")

print("Plantilla validacion experta")
display(validation.head(20))

print("Plantilla stock min/max")
display(stock.head(20))

**Insumos para decision operativa:**

Estas plantillas muestran el paso pendiente para una decision mas automatizada: expertos, inventario, stock minimo y stock maximo deben cerrar el ciclo operativo.

In [ ]:
plan = (reports_dir / "decision_support_plan.md").read_text(encoding="utf-8")
print(plan)

**Reglas de uso y revision:**

El plan de decision define el uso correcto del modelo: automatizar productos confiables y enviar productos estacionales, inactivos o de alto error a revision humana.

Para usarlo en toma de decisiones:

1. Revisar los productos con error alto.
2. Validar predicciones con expertos de produccion y comercial.
3. Completar stock actual, stock minimo y stock maximo.
4. Calcular una cantidad ajustada antes de emitir ordenes finales.